In [1]:
import time
import numpy as np, pandas as pd
from itertools import combinations
from sklearn.metrics import roc_auc_score

PARQUET_DIR = "./parquet"          # thư mục chứa DKT_{A,B,C}_f{fold}_s{seed}.parquet
SEED_DATA = 42
N_FOLDS = 5
N_BOOT = 1000

def paired_bootstrap_auc_fast(qa, qb, n_boot=1000, seed=0):
    m = qa[["qidx", "uid", "y_true", "p_late_mean"]].merge(
        qb[["qidx", "p_late_mean"]], on="qidx", suffixes=("_a", "_b"))
    assert len(m) == len(qa) == len(qb), "hai lần chạy phải có cùng tập câu hỏi"
    assert not m[["p_late_mean_a", "p_late_mean_b"]].isna().any().any(), "preds có NaN"
    y = m["y_true"].to_numpy().astype(np.float64)
    ucode, _ = pd.factorize(m["uid"])
    n_u = ucode.max() + 1

    def prep(p):
        _, inv = np.unique(p, return_inverse=True)
        return inv, inv.max() + 1
    ia, ga = prep(m["p_late_mean_a"].to_numpy())
    ib, gb = prep(m["p_late_mean_b"].to_numpy())

    def wauc(inv, g, w):
        wp = np.bincount(inv, weights=w * y, minlength=g)
        wn = np.bincount(inv, weights=w * (1 - y), minlength=g)
        cn = np.cumsum(wn) - wn
        return (wp * (cn + 0.5 * wn)).sum() / (wp.sum() * wn.sum())

    ones = np.ones(len(y))
    d0 = wauc(ib, gb, ones) - wauc(ia, ga, ones)
    rng = np.random.default_rng(seed)
    ds = np.empty(n_boot)
    for i in range(n_boot):
        counts = np.bincount(rng.integers(0, n_u, n_u), minlength=n_u)
        w = counts[ucode].astype(np.float64)
        ds[i] = wauc(ib, gb, w) - wauc(ia, ga, w)
    lo, hi = np.percentile(ds, [2.5, 97.5])
    return d0, lo, hi

rows, auc_rows = [], []
for fold in range(N_FOLDS):
    runs = {k: pd.read_parquet(f"{PARQUET_DIR}/DKT_{k}_f{fold}_s{SEED_DATA}.parquet")
            for k in "ABC"}

    # AUC từng chế độ + kiểm tra bản nhanh khớp sklearn
    aucs = {k: roc_auc_score(q["y_true"], q["p_late_mean"]) for k, q in runs.items()}
    auc_rows.append({"fold": fold, **{f"AUC_{k}": v for k, v in aucs.items()}})
    d0_chk = paired_bootstrap_auc_fast(runs["A"], runs["B"], n_boot=5)[0]
    assert abs(d0_chk - (aucs["B"] - aucs["A"])) < 1e-9, f"fold {fold}: d0 không khớp sklearn"

    for x, y in combinations("ABC", 2):      # B−A, C−A, C−B
        t = time.time()
        d0, lo, hi = paired_bootstrap_auc_fast(runs[x], runs[y], n_boot=N_BOOT, seed=0)
        print(f"fold {fold}: {y} − {x} xong sau {time.time()-t:.0f}s")
        rows.append({"fold": fold, "so sánh": f"{y} − {x}", "ΔAUC": d0,
                     "CI_low": lo, "CI_high": hi,
                     "khác biệt có ý nghĩa": (lo > 0) or (hi < 0)})

print("\n=== AUC từng chế độ ===")
print(pd.DataFrame(auc_rows).round(4).to_string(index=False))

df = pd.DataFrame(rows)
print("\n=== ΔAUC theo fold ===")
print(df.round(4).to_string(index=False))
df.to_csv("bootstrap_results_s42.csv", index=False)

# Kết luận theo tiêu chí: cả 5 khoảng tin cậy đều không chứa 0 (cùng dấu dương)
print("\n=== Kết luận ===")
for name, meaning in [("B − A", "phần do chọn checkpoint"),
                      ("C − B", "phần do cách huấn luyện"),
                      ("C − A", "tổng cải thiện")]:
    s = df[df["so sánh"] == name]
    ok = bool((s["CI_low"] > 0).all())
    print(f"{name} ({meaning}): ΔAUC TB = {s['ΔAUC'].mean():.4f}, "
          f"cả {N_FOLDS} CI > 0: {ok}")

fold 0: B − A xong sau 55s
fold 0: C − A xong sau 55s
fold 0: C − B xong sau 59s
fold 1: B − A xong sau 54s
fold 1: C − A xong sau 49s
fold 1: C − B xong sau 51s
fold 2: B − A xong sau 49s
fold 2: C − A xong sau 53s
fold 2: C − B xong sau 50s
fold 3: B − A xong sau 49s
fold 3: C − A xong sau 52s
fold 3: C − B xong sau 52s
fold 4: B − A xong sau 51s
fold 4: C − A xong sau 52s
fold 4: C − B xong sau 50s

=== AUC từng chế độ ===
 fold  AUC_A  AUC_B  AUC_C
    0 0.7838 0.7841 0.7891
    1 0.7842 0.7842 0.7890
    2 0.7842 0.7836 0.7892
    3 0.7844 0.7842 0.7891
    4 0.7838 0.7839 0.7889

=== ΔAUC theo fold ===
 fold so sánh    ΔAUC  CI_low  CI_high  khác biệt có ý nghĩa
    0   B − A  0.0003  0.0001   0.0005                  True
    0   C − A  0.0053  0.0048   0.0057                  True
    0   C − B  0.0050  0.0046   0.0054                  True
    1   B − A -0.0000 -0.0002   0.0002                 False
    1   C − A  0.0048  0.0044   0.0052                  True
    1   C − B  0.0